# Laya fine-tuning data — build the Kaggle folder `laya_v1`

Runs `backend/laya_datagen/laya_dataset.py` and shows what it wrote. Why each choice: `docs/laya-finetune/data.md`;
training it on Kaggle: `docs/laya-finetune/kaggle-guide.md` + `notebooks/laya_finetune_kaggle.ipynb`.

| Part | What it shows |
|---|---|
| **1 · Labels** | every usable label, by lecture set and source |
| **2 · Split** | which lectures are train / val / test (by whole lecture) |
| **3 · Build** | writes `data/laya_datagen/datasets/laya_v1/` (3 variants + shared test) and its counts |
| **4 · Examples** | three real rows, as Laya receives them |
| **5 · Soft labels** | ε per source and question, and what a target looks like |
| **6 · Token budget** | how long the inputs are vs `max_len = 1024`, and what gets cut |
| **7 · Shortcut check** | how far span length alone gets (the floor Laya must beat) |
| **8 · Validation** | the files loaded with Laya's own reader, nothing skipped |

Nothing here calls a paid model or trains anything.

## Setup

In [1]:
import sys, os, json, importlib
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "backend").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from IPython.display import display, Markdown
from backend.laya_datagen import export, laya_dataset as L
for _m in (export, L): importlib.reload(_m)
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)

---
## 1 · Labels
One label per finding (you > Bin Baz written text > Sonnet high > Sonnet medium; JEV / Gemini / base Laya are never
labels). Takes ~20 s: it re-reads every lecture's findings.

In [2]:
inv = L.load_inventory()
recs, where, ans, usable = inv
df = pd.DataFrame([{"set": x["set"], "source": x["source"], "verdict": x["verdict"]} for x in usable])
display(df.pivot_table(index=["set", "source"], columns="verdict", aggfunc="size", fill_value=0))

verdict                                                       CORRECT  FALSE  NOT_SURE  PARAPHRASE
set                         source                                                                
lectures batch 1            Automatic (Bin Baz written text)       62      1         0           0
                            Sonnet 5.5 — high                     746    627         0          63
                            Sonnet 5.5 — medium                    84     11         0           5
lectures batch 2            Automatic (Bin Baz written text)       66      0         0           0
                            Sonnet 5.5 — high                     928    955         0          86
review (15 Bin Baz lessons) Automatic (Bin Baz written text)      436     17         0           0
                            Sonnet 5.5 — high                     302    573         0          44
                            You — negatives sprint                 57     29         1          24
                            You — stages 2 / 4                    282     26         0          21

---
## 2 · Split — by whole lecture
* **test** = 6 review lessons (your 198 labels; the other 9 lessons hold your other 197, which are trained on) + 1 lecture
  per YouTube sheikh + 2 Bin Baz lectures from the batches (Sonnet's labels on varied speakers).
* **val** = 1 lecture of every other sheikh + 2 Bin Baz lectures: fits the calibration temperature, watches overfitting.
* **train** = the rest. The rule uses only the lecture lists, never the labels, so it does not move when labels change.

In [3]:
info = L.lecture_info(where)
split = L.assign_splits(info)
rows_all = [x for x in usable if x["exportable"]]
lec = pd.DataFrame([{"lecture": d, **info[d], "split": split[d],
                     "labels": sum(1 for x in rows_all if x["doc"] == d),
                     "yours": sum(1 for x in rows_all if x["doc"] == d and x["source"].startswith("You"))} for d in info])
display(lec.pivot_table(index="split", columns="group", values="lecture", aggfunc="count", fill_value=0))
display(lec[lec.split != "train"].sort_values(["split", "group", "speaker"]).reset_index(drop=True))

group,lectures,review
split,,
test,14,6
train,48,9
val,9,0


,lecture,group,batch,speaker,split,labels,yours
0,yt_VNEh4-Sjg3U,lectures,1,abdulrazzaq_al_badr,test,44,0
1,yt_u3hNd4jL0yQ,lectures,1,abdussalam_al_shuwayer,test,49,0
2,yt_0NpOzXjhhHw,lectures,2,al_albani,test,24,0
3,yt_gFCif1bK1_U,lectures,2,al_fawzan,test,94,0
4,yt_HgYC6n5vVvk,lectures,2,aziz_farhan_al_anzi,test,33,0
5,binbaz_audio_191,lectures,2,binbaz,test,16,0
6,binbaz_audio_193,lectures,2,binbaz,test,40,0
7,yt_xiXs8sApQG0,lectures,1,ibn_uthaymeen,test,94,0
8,yt_DXR7qH70ySM,lectures,2,mohammed_meziane,test,64,0
9,yt_ntevs6C31-M,lectures,2,saeed_ruslan,test,59,0


---
## 3 · Build the folder
Three variants share the same `val` and `test`, so their results are directly comparable:
`soft_asis` (soft labels, as is) · `soft_balanced` (soft, "not a quote" downsampled to 25 % in train) · `hard_asis` (one-hot).
The Bin Baz written-text labels are **test-only**: in train / val those findings are dropped.

In [4]:
m = L.build(inv=inv)
print("written to", L.OUT)
display(pd.DataFrame([{"file": f, "rows": n} for f, n in m["files"].items()]))
print("dropped:", m["dropped"])
for v, d in m["variants"].items():
    display(Markdown(f"**{v}** — {d['about']} — train 'not a quote' share **{d['train_false_share']:.1%}**"))
    display(pd.DataFrame(d["train"]).set_index("source"))
display(Markdown("**val** (same for the three variants)")); display(pd.DataFrame(m["variants"]["soft_asis"]["val"]).set_index("source"))
display(Markdown("**test** by label source")); display(pd.DataFrame(m["test"]["by source"]).set_index("source"))
display(Markdown("**test** by speaker")); display(pd.DataFrame(m["test"]["by speaker"]).set_index("speaker"))

written to /home/zeno/Documents/work/new_projects/Wahy-Transcriber/data/laya_datagen/datasets/laya_v1


,file,rows
0,soft_asis/train.jsonl,2863
1,soft_asis/val.jsonl,515
2,soft_balanced/train.jsonl,1965
3,soft_balanced/val.jsonl,515
4,hard_asis/train.jsonl,2863
5,hard_asis/val.jsonl,515
6,test.jsonl,1523
7,test_user.jsonl,198


dropped: {"quoted by meaning (not an option of Laya's question)": 243, 'written label in train (test-only source)': 272, 'written label in val (test-only source)': 30}


**soft_asis** — soft labels (eps per source and question), training data as is — train 'not a quote' share **48.5%**

,rows,real,not a quote,unsure,% not a quote,with boundary,complete_span,missing_start,missing_end,missing_both
source,,,,,,,,,,
sonnet_high,2596,1242,1354,0,52.2,1241,1018,125,86,12
sonnet_medium,70,63,7,0,10.0,63,57,4,2,0
user,197,168,28,1,14.2,139,87,20,24,8


...

,rows,real,not a quote,unsure,% not a quote,with boundary,complete_span,missing_start,missing_end,missing_both
source,,,,,,,,,,
sonnet_high,508,259,249,0,49.0,259,221,21,16,1
sonnet_medium,7,5,2,0,28.6,5,5,0,0,0


**test** by label source

,rows,real,not a quote,unsure,% not a quote,with boundary,complete_span,missing_start,missing_end,missing_both
source,,,,,,,,,,
sonnet_high,1027,475,552,0,53.7,475,372,49,49,5
sonnet_medium,18,16,2,0,11.1,16,13,2,1,0
user,198,171,27,0,13.6,143,64,23,36,20
written,280,274,6,0,2.1,271,169,43,39,20


**test** by speaker

,rows,real,not a quote,unsure,% not a quote,with boundary,complete_span,missing_start,missing_end,missing_both
speaker,,,,,,,,,,
abdulrazzaq_al_badr,44,20,24,0,54.5,20,17,1,2,0
abdussalam_al_shuwayer,49,30,19,0,38.8,30,25,3,2,0
al_albani,24,15,9,0,37.5,15,5,3,6,1
al_fawzan,94,56,38,0,40.4,56,48,6,2,0
aziz_farhan_al_anzi,33,15,18,0,54.5,15,15,0,0,0
binbaz,958,616,342,0,35.7,585,356,87,98,44
ibn_uthaymeen,94,77,17,0,18.1,77,64,7,6,0
mohammed_meziane,64,32,32,0,50.0,32,26,4,2,0
saeed_ruslan,59,21,38,0,64.4,21,16,3,2,0


---
## 4 · Three real rows, as Laya receives them
`state` = the envelope the app builds (`WahyAiVerifier.buildCanonicalDecisionEnvelope`, `js/ai-verifier.js`) minus the
engine's own score / band. `gold` = the target probabilities. `meta` is ours: the trainer ignores it.

In [5]:
test = [json.loads(l) for l in open(L.OUT / "test.jsonl", encoding="utf-8")]
train = [json.loads(l) for l in open(L.OUT / "soft_asis" / "train.jsonl", encoding="utf-8")]
pick = [next(r for r in test if r["meta"]["source"] == "user" and r["meta"].get("boundary") == "MISSING_END"),
        next(r for r in train if r["meta"]["verdict"] == "FALSE" and r["meta"]["span_words"] == 2),
        next(r for r in train if r["meta"]["group"] == "lectures" and r["meta"]["source_type"] == "HADITH" and r["meta"]["verdict"] == "CORRECT")]
for r in pick:
    display(Markdown(f"#### {r['meta']['source']} · {r['meta']['verdict']} · {r['meta']['boundary']} · {r['meta']['speaker']} ({r['meta']['split']})"))
    print(json.dumps({"state": r["state"], "gold": r["gold"], "meta": r["meta"]}, ensure_ascii=False, indent=1))

#### user · CORRECT · MISSING_END · binbaz (test)

{
 "state": {
  "source_type": "QURAN",
  "candidate_reference": "البقرة: 43",
  "canonical_corpus_text": "وَأَقِيمُوا الصَّلَاةَ وَآتُوا الزَّكَاةَ وَارْكَعُوا",
  "context_before": "تأخذ إلى يردين. أما بعد أيها المستمعون الكرام، هذه معكم اليوم في وجود إقامة الصلاوات الخمس ونحمد عليها. يقول الله جل وعلا في فتابه المبين،",
  "matched_span_in_lecture": "وأقوموا الصلاة وآتوا الزكاة واركعوه",
  "context_after": "عن راكعين. في آيات كثيرات نبرفها سبحانه في إقامة الصلاوات. وما ذاك من الوقت من شعرها، وكأنها عمود الإسلام، كما جاء في الحديث عنه عليه",
  "unmatched_corpus_prefix_words": "",
  "immediate_transcript_before": "عليها. يقول الله جل وعلا في فتابه المبين،",
  "unmatched_corpus_suffix_words": "مَعَ الرَّاكِعِينَ",
  "immediate_transcript_after": "عن راكعين. في آيات كثيرات نبرفها سبحانه في",
  "quran_neighboring_context": {
   "surah": 2,
   "ayah": 43,
   "previous_ayah_text": "وَلَا تَلْبِسُوا الْحَقَّ بِالْبَاطِلِ وَتَكْتُمُوا الْحَقَّ وَأَنتُمْ تَعْلَمُونَ",
   "current_ayah_text": "

#### sonnet_high · FALSE · None · binbaz (train)

{
 "state": {
  "source_type": "QURAN",
  "candidate_reference": "الرعد: 17",
  "canonical_corpus_text": "الْحَقَّ وَالْبَاطِلَ",
  "context_before": "هذا إلا بالعناية في كتاب الله وسنة الرسول صلى الله عليه وسلم، فمنهما تعرف أحكام الله، وتعرف سنة رسوله عليه صلى الله عليه وسلم، ويره",
  "matched_span_in_lecture": "الحق والباطل،",
  "context_after": "فالواجب على كل مسلم أن يتفقه في دينه، وأن يتبصر في ما يجب عليه، يقول النبي صلى الله عليه وسلم من يريد الله بخير يفقه",
  "unmatched_corpus_prefix_words": "النَّارِ ابْتِغَاءَ حِلْيَةٍ أَوْ مَتَاعٍ زَبَدٌ مِّثْلُهُ كَذَٰلِكَ يَضْرِبُ اللَّهُ",
  "immediate_transcript_before": "سنة رسوله عليه صلى الله عليه وسلم، ويره",
  "unmatched_corpus_suffix_words": "فَأَمَّا الزَّبَدُ فَيَذْهَبُ جُفَاءً وَأَمَّا مَا يَنفَعُ النَّاسَ فَيَمْكُثُ فِي",
  "immediate_transcript_after": "فالواجب على كل مسلم أن يتفقه في دينه،",
  "quran_neighboring_context": {
   "surah": 13,
   "ayah": 17,
   "previous_ayah_text": "لِلَّهِ شُرَكَاءَ خَلَقُوا كَخَلْقِهِ فَتَشَاب

#### sonnet_high · CORRECT · MISSING_END · binbaz (train)

{
 "state": {
  "source_type": "HADITH",
  "candidate_reference": "صحيح البخاري (1550)",
  "canonical_corpus_text": "يُلَبِّي لَبَّيْكَ اللَّهُمَّ لَبَّيْكَ لَبَّيْكَ لاَ شَرِيكَ لَكَ لَبَّيْكَ إِنَّ الْحَمْدَ",
  "context_before": "ويوصل ركاتين، ثم يلبيه بالحج أو بالعمرة، فيقول اللهم لبك عمره إن كان عمره، أو اللهم لبك حجة إن كان حجة، من ميقات بلد، ثم",
  "matched_span_in_lecture": "يلبي التلبية الشرعية، لبك اللهم لبيك، لبيك لا شريك لك لبيك، إن الحمد",
  "context_after": "وإن مسه لك ومك لا شريك لك، يكرر هذه التلبية حتى يصل إلى الحرم، ويكررها في طريق، وحتى المرأة الحائض والنفساء تحرمان من المقهات بالهج",
  "unmatched_corpus_prefix_words": "تْ إِنِّي لأَعْلَمُ كَيْفَ كَانَ النَّبِيُّ صلى الله عليه وسلم",
  "immediate_transcript_before": "حجة إن كان حجة، من ميقات بلد، ثم",
  "unmatched_corpus_suffix_words": "وَالنِّعْمَةَ لَكَ تَابَعَهُ أَبُو مُعَاوِيَةَ عَنِ الأَعْمَشِ وَقَالَ شُعْبَةُ أَخْبَرَنَا",
  "immediate_transcript_after": "وإن مسه لك ومك لا شريك لك، يكرر",
  "quran_neighboring_c

---
## 5 · Soft labels
ε = probability moved off the labelled answer, spread over the other options; ≈ how often that source disagrees with you
(written text: 155/156 verdicts but 128/152 boundaries; Sonnet high, blind: 58/60 and 23/24). `hard_asis` uses ε = 0.

In [6]:
display(pd.DataFrame(export.EPS_BY_SOURCE).T)
print("Sonnet high, 'not a quote':", export.soft(export.VERDICTS, "FALSE", export.EPS_BY_SOURCE["sonnet_high"]["verdict"]))
print("you, 'missing end':        ", export.soft(export.BOUNDARIES, "MISSING_END", export.EPS_BY_SOURCE["user"]["boundary_status"]))

,verdict,boundary_status
user,0.02,0.05
written,0.03,0.15
sonnet_high,0.08,0.10
sonnet_medium,0.12,0.15


Sonnet high, 'not a quote': {'CORRECT': 0.04, 'FALSE': 0.92, 'NOT_SURE': 0.04}
you, 'missing end':         {'COMPLETE_SPAN': 0.0167, 'MISSING_START': 0.0167, 'MISSING_END': 0.95, 'MISSING_BOTH': 0.0167}


---
## 6 · Token budget
Laya reads `[CLS] question [SEP] [MASK] option … [SEP] state-as-JSON [SEP]` up to `max_len = 1024` tokens, and keeps the
**start** of the state when it is too long (training and the app alike). The verdict question takes 178 tokens, the
boundary question 242. Downloads the tokenizer once (~1 GB checkpoint, cached).

In [7]:
tok, cfg, model_dir = L.tokenizer()
tr = pd.DataFrame(L.truncation(train + test, tok))
print(tr["state_tokens"].describe(percentiles=[.5, .9, .95, .99]).round(0).to_dict())
display(tr[[c for c in tr.columns if c.startswith(("span cut", "state cut"))]].sum().rename("rows").to_frame())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

{'count': 4386.0, 'mean': 486.0, 'std': 143.0, 'min': 210.0, '50%': 480.0, '90%': 631.0, '95%': 690.0, '99%': 889.0, 'max': 3180.0}


,rows
span cut (verdict),15
state cut (verdict),55
state cut (boundary),87


---
## 7 · Shortcut check — how far do cheap fields get?
A logistic regression fit on train (real vs not a quote) using only the span length, or 7 cheap state fields. Span length is
real evidence (two-word spans are mostly coincidences) but it is a trap for short real quotes like «وَالْعَصْرِ»: Laya has to
beat this line, especially on short spans.

In [8]:
sys.path.insert(0, str(L.OUT)); import eval_laya; importlib.reload(eval_laya)
base = json.load(open(L.OUT / "baselines.json", encoding="utf-8"))
P = eval_laya.baseline_predictions(base)
res = [eval_laya.evaluate(test, P[n], n) for n in P]
for g in ["you", "you + Sonnet", "Sonnet · other lectures"]:
    display(Markdown(f"**{g}**\n\n" + eval_laya.compare(res, g)))
tdf = pd.DataFrame([{"span words": min(r["meta"]["span_words"], 10), "not a quote": r["meta"]["verdict"] == "FALSE"} for r in train])
display(tdf.groupby("span words")["not a quote"].agg(["size", "mean"]).rename(columns={"size": "train rows", "mean": "share not a quote"}).round(2).T)

**you**

| model | cards | verdict accuracy | not-a-quote recall | not-a-quote precision | real wrongly rejected | AUC | ECE | boundary accuracy |
|---|---|---|---|---|---|---|---|---|
| JEV (cached) | 147 | 86.4 % [80–91] | 20.8 % [9–40] | 100.0 % [57–100] | 0.0 % [0–3] | 0.604 | 0.136 | 52.9 % [43–62] |
| Gemini (cached) | 147 | 89.8 % [84–94] | 41.7 % [24–61] | 100.0 % [72–100] | 0.0 % [0–3] | 0.708 | 0.102 | 50.0 % [40–60] |
| Laya (base) (cached) | 111 | 54.0 % [45–63] | 88.9 % [56–98] | 14.0 % [7–25] | 48.0 % [39–58] | 0.704 | 0.460 | 30.4 % [22–40] |
| span length only (LR) | 198 | 79.3 % [73–84] | 29.6 % [16–48] | 26.7 % [14–44] | 12.9 % [9–19] | 0.723 | 0.109 | — |
| 7 shallow fields (LR) | 198 | 83.8 % [78–88] | 66.7 % [48–81] | 43.9 % [30–59] | 13.5 % [9–19] | 0.857 | 0.040 | — |

**you + Sonnet**

| model | cards | verdict accuracy | not-a-quote recall | not-a-quote precision | real wrongly rejected | AUC | ECE | boundary accuracy |
|---|---|---|---|---|---|---|---|---|
| JEV (cached) | 180 | 87.2 % [82–91] | 19.2 % [9–38] | 83.3 % [44–97] | 0.7 % [0–4] | 0.593 | 0.128 | 54.9 % [46–63] |
| Gemini (cached) | 180 | 90.0 % [85–94] | 38.5 % [22–57] | 90.9 % [62–98] | 0.7 % [0–4] | 0.689 | 0.100 | 51.9 % [43–60] |
| Laya (base) (cached) | 111 | 54.0 % [45–63] | 88.9 % [56–98] | 14.0 % [7–25] | 48.0 % [39–58] | 0.704 | 0.460 | 30.4 % [22–40] |
| span length only (LR) | 1243 | 81.7 % [79–84] | 78.7 % [75–82] | 81.5 % [78–84] | 15.7 % [13–19] | 0.890 | 0.042 | — |
| 7 shallow fields (LR) | 1243 | 85.5 % [83–87] | 89.3 % [87–92] | 81.5 % [78–84] | 17.8 % [15–21] | 0.932 | 0.035 | — |

**Sonnet · other lectures**

| model | cards | verdict accuracy | not-a-quote recall | not-a-quote precision | real wrongly rejected | AUC | ECE | boundary accuracy |
|---|---|---|---|---|---|---|---|---|
| span length only (LR) | 596 | 81.9 % [79–85] | 80.9 % [76–85] | 79.1 % [74–84] | 17.3 % [14–22] | 0.897 | 0.051 | — |
| 7 shallow fields (LR) | 596 | 86.6 % [84–89] | 91.8 % [88–94] | 80.9 % [76–85] | 17.6 % [14–22] | 0.941 | 0.057 | — |

span words,1,2,3,4,5,6,7,8,9,10
train rows,2.0,534.00,468.00,378.00,272.00,198.00,137.00,116.00,80.00,678.00
share not a quote,0.0,0.92,0.81,0.55,0.44,0.31,0.26,0.25,0.08,0.09


---
## 8 · Validation with Laya's own reader
`laya.train.read_jsonl` + `items_from_rows` at the training budgets: every row must give a verdict item, every real quote
with a boundary a boundary item, nothing skipped. Then the trainer's `dry_run` (no weights loaded).

In [9]:
display(pd.DataFrame(L.validate(tok=tok, cfg=cfg)))
from laya.train import dry_run, TrainConfig
s = dry_run(str(L.OUT / "soft_asis" / "train.jsonl"), model_dir, TrainConfig(max_len=1024, head_max_len=256))

,file,rows,items,expected items,skipped,ok
0,hard_asis/train.jsonl,2863,4306,4306,{},True
1,hard_asis/val.jsonl,515,779,779,{},True
2,soft_asis/train.jsonl,2863,4306,4306,{},True
3,soft_asis/val.jsonl,515,779,779,{},True
4,soft_balanced/train.jsonl,1965,3408,3408,{},True
5,soft_balanced/val.jsonl,515,779,779,{},True
6,test.jsonl,1523,2428,2428,{},True
7,test_user.jsonl,198,341,341,{},True


dry-run: 2863 rows read, 4306 valid items, skipped: {}


dry-run: optimizer budget: 3906 train item(s), effective batch up to 64 (micro_batch 8 x grad_accum 8), 248 optimizer updates over 4 epoch(s)
